# Итоговый отчёт по модели SIR

**Авторы:** А. В. Королькова, PhD, Кулябов Д. С., DSc

**Принадлежность:** Российский университет дружбы народов

## Назначение скрипта

Данный скрипт загружает ранее сохранённые результаты моделирования SIR
и строит сравнительные графики для итогового отчёта.

### Что делает скрипт

1. Загружает результаты трёх предыдущих экспериментов:
   - Детерминированная симуляция (`sir_det.csv`)
   - Стохастическая симуляция (`sir_stoch.csv`)
   - Сканирование параметра β (`sir_scan.csv`)

2. Строит два итоговых графика:
   - Сравнение детерминированной и стохастической динамики I(t)
   - Зависимость пика инфицированных от коэффициента заражения β

## Входные данные

| Файл | Создан скриптом | Описание |
|------|-----------------|----------|
| `data/sir_det.csv` | Базовый прогон | Детерминированная симуляция |
| `data/sir_stoch.csv` | Базовый прогон | Стохастическая симуляция |
| `data/sir_scan.csv` | Сканирование β | Результаты параметрического анализа |

**Важно:** Эти файлы должны быть созданы до запуска данного скрипта.

## Выходные данные

| Файл | Описание | Рисунок |
|------|----------|---------|
| `plots/comparison.png` | Сравнение I(t) детерминированной и стохастической симуляций | Рис. 6.5 |
| `plots/sensitivity.png` | Зависимость пика I от коэффициента β | Рис. 6.6 |

## Интерпретация результатов

### Рисунок 6.5: Сравнение детерминированной и стохастической динамики

- При большом размере популяции (N = 1000) стохастическая траектория
  близка к детерминированной

- Стохастическая кривая может иметь небольшие флуктуации

- Различия между кривыми демонстрируют влияние случайности
  на динамику эпидемии

### Рисунок 6.6: Зависимость пика I от β (чувствительность)

- Позволяет количественно оценить, как изменение заразности β
  влияет на тяжесть эпидемии

- Важно для принятия решений (например, оценка эффекта от мер
  по снижению β: карантин, маски, социальная дистанция)

- Демонстрирует нелинейный характер зависимости: малые изменения β
  в критической области приводят к значительным изменениям пика I

## Инициализация проекта DrWatson

In [ ]:
using DrWatson
@quickactivate "project"

## Подключение утилит для работы с данными и графикой

In [ ]:
using DataFrames, CSV, Plots

## Загрузка результатов экспериментов

### Загрузка детерминированной симуляции

Файл содержит колонки: time, S, I, R
Параметры симуляции: β = 0.3, γ = 0.1, tmax = 100.0

In [ ]:
df_det = CSV.read(datadir("sir_det.csv"), DataFrame)

### Загрузка стохастической симуляции

Файл содержит колонки: time, S, I, R
Параметры симуляции: β = 0.3, γ = 0.1, tmax = 100.0
Зерно ГСЧ: 123 (воспроизводимость)

In [ ]:
df_stoch = CSV.read(datadir("sir_stoch.csv"), DataFrame)

### Загрузка результатов сканирования β

Файл содержит колонки: β, peak_I, final_R
Диапазон β: 0.1 : 0.05 : 0.8

In [ ]:
df_scan = CSV.read(datadir("sir_scan.csv"), DataFrame)

## Построение итоговых графиков

### Рисунок 6.5: Сравнение детерминированной и стохастической динамики

На одном графике отображаются две кривые I(t):
- **Deterministic I** (синяя линия) — усреднённая динамика из ОДУ
- **Stochastic I** (красная линия) — траектория из алгоритма Гиллеспи

Особенности:
- Стохастическая кривая обрезается до длины детерминированной
  для корректного сравнения
- Оси: время по горизонтали, число инфицированных по вертикали

In [ ]:
p1 = plot(
    df_det.time,
    [df_det.I df_stoch.I[1:length(df_det.time)]],
    label = ["Deterministic I" "Stochastic I"],
    xlabel = "Time",
    ylabel = "Infected",
    title = "Comparison",
)

savefig(plotsdir("comparison.png"))

### Рисунок 6.6: Зависимость пика I от β (чувствительность)

График отображает зависимость максимального числа инфицированных
от коэффициента заражения β.

Характерные особенности:
- Пороговый эффект: при малых β эпидемия не развивается (peak I ≈ 0)
- Резкий нелинейный рост в критической области
- Насыщение при больших β (почти всё население переболевает)

In [ ]:
p2 = plot(
    df_scan.β,
    df_scan.peak_I,
    marker = :circle,
    xlabel = "β",
    ylabel = "Peak I",
    title = "Sensitivity",
)

savefig(plotsdir("sensitivity.png"))

## Завершение работы

Скрипт успешно выполнил:
- Загрузка трёх CSV-файлов с результатами
- Построение графика сравнения → `plots/comparison.png`
- Построение графика чувствительности → `plots/sensitivity.png`

In [ ]:
println("Отчётные графики сохранены в plots/")